In [13]:
import sqlite3

def list_sqlite_sessions(db_path: str):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Assuming a table where sessions/checkpoints are stored and have a thread_id
    # You might need to inspect your 'chat_memory.db' to find the actual table and column names
    cursor.execute("SELECT DISTINCT thread_id FROM checkpoints ORDER BY checkpoint_id DESC")
    sessions = cursor.fetchall()
    conn.close()
    return [session[0] for session in sessions]

In [14]:
sessions = list_sqlite_sessions("/home/web-h-063/Documents/auto-data-intelligence/chat_memory.db")

In [20]:
from langgraph.checkpoint.sqlite import SqliteSaver # Or AsyncSqliteSaver for async
from langgraph.graph import StateGraph, MessagesState, START
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

# Assume you have a simple graph definition for demonstration
# In your actual application, this would be your compiled graph
def create_dummy_graph():
    builder = StateGraph(MessagesState)
    builder.add_node("chatbot", lambda state: {"messages": [AIMessage("Hello!")]})
    builder.add_edge(START, "chatbot")
    return builder.compile()

# Initialize your saver (replace with AsyncSqliteSaver if you're using async)
# For this example, we'll use SqliteSaver for synchronous loading
saver = SqliteSaver.from_conn_string("/home/web-h-063/Documents/auto-data-intelligence/chat_memory.db")

# List of sessions obtained from your previous query
sessions = [
    'session_e4710e49', 'session_b9ec3106', 'session_de9d8c9a'
]

async def inspect_sessions(db_path: str, session_ids: list[str]):
    async with AsyncSqliteSaver.from_conn_string(db_path) as saver:
        for session_id in session_ids:
            print(f"\n--- Content for Session: {session_id} ---")
            try:
                config = {"configurable": {"thread_id": session_id}}
                checkpoint = await saver.aget(config) # Use await for async get

                if checkpoint and 'channel_values' in checkpoint and 'messages' in checkpoint['channel_values']:
                    messages = checkpoint['channel_values']['messages']
                    for msg in messages:
                        print(f"{msg.type.capitalize()}: {msg.content}")
                else:
                    print("No messages found for this session or checkpoint format is unexpected.")
            except Exception as e:
                print(f"Error loading session {session_id}: {e}")

# Run the async function
await inspect_sessions("/home/web-h-063/Documents/auto-data-intelligence/chat_memory.db", sessions)


--- Content for Session: session_e4710e49 ---
Human: whats up with order 2010099827? can you help me please?
Ai: [{'signature': '6c6e60b1ad5d2b999391a218bf69392e8a0f8025a9a91e5f587aa3543e390e44', 'thinking': "The user is asking about a specific order - order number 2010099827. They want to know what's going on with this order. This is a data inquiry, so I should follow my workflow:\n\n1. First, recall similar past queries\n2. Fetch schema and business context\n3. Compose SQL targeting Wren model names\n4. Execute the query\n\nLet me start by recalling similar queries and fetching context.\n", 'type': 'thinking'}, {'id': 'call_function_1admqtvta4mg_1', 'input': {'limit': 3, 'question': 'order status details'}, 'name': 'wren_recall_queries', 'type': 'tool_use'}, {'id': 'call_function_1admqtvta4mg_2', 'input': {'item_type': 'model', 'question': 'order details status information'}, 'name': 'wren_fetch_context', 'type': 'tool_use'}]
Tool: {"ok": false, "content": "Error embedding content (